In [1]:
import anndata as ad
import pandas as pd
import numpy as np

import umap
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import calinski_harabasz_score

import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
input_dir = "/Users/apple/Desktop/KB/data"
adata_train = ad.read_h5ad(input_dir+'/BiddyData/Biddy_train.h5ad')
adata_test = ad.read_h5ad(input_dir+'/BiddyData/Biddy_test.h5ad')

train_labels = adata_train.obs["clone_id"].to_numpy()
test_labels = adata_test.obs["clone_id"].to_numpy()


train_embeddings = np.load(input_dir+'/feat_LCL_2025/cell_tag/scvi_embedding/Biddy_scvi_train_latent32.npy')
test_embeddings = np.load(input_dir+'/feat_LCL_2025/cell_tag/scvi_embedding/Biddy_scvi_test_latent32.npy')

print(train_labels.shape, test_labels.shape)
print(train_embeddings.shape, test_embeddings.shape)

(5893,) (641,)
(5893, 32) (641, 32)


#### calinski score

In [3]:
# Calculate the Calinski-Harabasz score
score = calinski_harabasz_score(train_embeddings, train_labels)

# Print the score
print("Train Calinski-Harabasz Score:", score)


Train Calinski-Harabasz Score: 10.073846917837068


In [4]:
# Calculate the Calinski-Harabasz score
score = calinski_harabasz_score(test_embeddings, test_labels)

# Print the score
print("Test Calinski-Harabasz Score:", score)


Test Calinski-Harabasz Score: 2.9647068387237714


## KNN classifier

In [5]:
adata_train.obs["clone_id"].value_counts()

clone_id
493.0     1178
2352.0     591
487.0      329
666.0      296
2721.0     263
          ... 
2902.0       5
2951.0       5
2863.0       5
2894.0       5
2367.0       5
Name: count, Length: 169, dtype: int64

### Test Accuracy

In [6]:

# Initialize the KNN classifier (you can adjust the number of neighbors)
knn = KNeighborsClassifier(n_neighbors=5)

# Train the KNN classifier
knn.fit(train_embeddings, train_labels)

# Predict the labels for the test set
y_pred = knn.predict(test_embeddings)

# Calculate the accuracy
accuracy = accuracy_score(test_labels, y_pred)

print(f"KNN classifier testing accuracy: {accuracy * 100:.2f}%")


KNN classifier testing accuracy: 52.73%


### Train Accuracy

In [7]:
# Split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(train_embeddings, train_labels, test_size=0.2, random_state=42)

# Initialize the KNN classifier (you can adjust the number of neighbors)
knn = KNeighborsClassifier(n_neighbors=5)

# Train the KNN classifier
knn.fit(X_train, y_train)

# Predict the labels for the test set
y_pred = knn.predict(X_test)

# Calculate the accuracy
accuracy = accuracy_score(y_test, y_pred)

print(f"KNN classifier training accuracy: {accuracy * 100:.2f}%")


KNN classifier training accuracy: 52.16%
